<a href="https://colab.research.google.com/github/Almas1989/SQL_vs_pandas/blob/main/EDA(SQL%2Cpandas).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Краткий практикум по Duck DB и Pandas (чтобы сравнить запросы SQL и код в pandas) </br>
некторой выборки по заработным платам в сфере IT в Казахстане**

In [6]:
!git clone https://github.com/Almas1989/SQL_vs_pandas.git
%cd SQL_vs_pandas

Cloning into 'SQL_vs_pandas'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 18 (delta 9), reused 17 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 152.80 KiB | 1.46 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/SQL_vs_pandas


In [4]:
# Загрузим необходимые библиотеки
import duckdb
import pandas as pd
import numpy as np
import re, io
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

In [7]:
# Сначала посмотрим на проблемную строку
with open('itkz_salaries.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()
    print("Строка 19:", lines[18])  # строки нумеруются с 0

Строка 19: 900000,3,2025,25,PreferNotToSay,2019,Fullstack Developer,Middle,Local,Almaty,Laravel,Торговля (FMCG, Retail),,2025-08-21 05:15:46



In [8]:

with open('itkz_salaries.csv', 'r', encoding='utf-8') as f:
    content = f.read()

lines = content.split('\n')
fixed_lines = []

for i, line in enumerate(lines):
    if line.strip():  # пропускаем пустые строки
        # Подсчитываем количество полей
        fields = line.count(',') + 1
        if fields > 14:  # если полей больше ожидаемого
            print(f"Проблемная строка {i+1}: {line}")
            # Находим лишние запятые и заменяем их на точку с запятой
            # Простой способ: заменить все запятые на точку с запятой в 12-м поле
            parts = line.split(',')
            if len(parts) > 12:
                # Склеиваем лишние части 12-го поля через точку с запятой
                fixed_field_12 = ';'.join(parts[11:-2])  # все между 12-м полем и последними двумя
                # Пересобираем строку
                line = ','.join(parts[:11] + [fixed_field_12] + parts[-2:])
    fixed_lines.append(line)

content = '\n'.join(fixed_lines)

# Загружаем исправленные данные
df = pd.read_csv(StringIO(content), encoding='utf-8')
print(f"Загружено {len(df)} строк")
print(df.head())

Проблемная строка 19: 900000,3,2025,25,PreferNotToSay,2019,Fullstack Developer,Middle,Local,Almaty,Laravel,Торговля (FMCG, Retail),,2025-08-21 05:15:46
Проблемная строка 164: 1250000,3,2025,30,Male,2020,Backend Developer,Middle,Local,Almaty,Python,Торговля (FMCG, Retail),,2025-07-02 06:42:51
Проблемная строка 201: 756000,2,2025,21,Male,2023,Backend Developer,Junior,Local,Astana,Golang,Торговля (FMCG, Retail),,2025-06-21 05:49:19
Проблемная строка 261: 1200000,2,2025,32,Male,2019,Backend Developer,Senior,Local,Almaty,Java,Торговля (FMCG, Retail),,2025-06-10 09:50:30
Проблемная строка 263: 1000000,2,2025,36,Male,2021,Backend Developer,Senior,Local,Almaty,.NET,Торговля (FMCG, Retail),,2025-06-10 08:15:15
Проблемная строка 286: 500000,2,2025,27,Male,2022,Маркетинговый аналитик,Middle,Local,Astana,,Торговля (FMCG, Retail),,2025-06-04 06:05:23
Проблемная строка 328: 2550000,2,2025,24,Male,2017,Frontend Developer,Senior,Foreign,Almaty,React,Торговля (FMCG, Retail),,2025-05-22 05:28:48
Проблем

In [9]:
df.head()

,Value,Quarter,Year,Age,Gender,Started,Profession,Grade,Company,City,Skill,Industry,SourceType,Created
0,800000,3,2025,29.0,Male,2022.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,NaN,2025-09-01 17:08:55
1,500000,3,2025,19.0,Male,2024.0,Backend Developer,Junior,Foreign,Almaty,Java,IT продукт,NaN,2025-08-29 20:02:53
2,1160000,1,2025,23.0,Male,2022.0,Frontend Developer,Senior,Local,Almaty,Javascript,E-Commerce,NaN,2025-08-28 15:57:36
3,500000,3,2025,20.0,Male,2023.0,Backend Developer,Junior,Local,Astana,Java,Государственное учреждение (Government),NaN,2025-08-28 11:44:12
4,1200000,3,2025,33.0,Male,2017.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,NaN,2025-08-28 06:07:45


In [22]:
# Основная информация о датафрейме
print("=== ОСНОВНАЯ ИНФОРМАЦИЯ ===")
print(f"dataframe shape {df.shape}")
print(f"список столбцов {list(df.columns)}")

=== ОСНОВНАЯ ИНФОРМАЦИЯ ===
dataframe shape (1422, 14)
список столбцов ['Value', 'Quarter', 'Year', 'Age', 'Gender', 'Started', 'Profession', 'Grade', 'Company', 'City', 'Skill', 'Industry', 'SourceType', 'Created']


In [12]:
print("\n=== ИНФОРМАЦИЯ О СТОЛБЦАХ ===")
print(df.info())


=== ИНФОРМАЦИЯ О СТОЛБЦАХ ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1422 entries, 0 to 1421
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Value       1422 non-null   int64  
 1   Quarter     1422 non-null   int64  
 2   Year        1422 non-null   int64  
 3   Age         641 non-null    float64
 4   Gender      936 non-null    object 
 5   Started     638 non-null    float64
 6   Profession  1385 non-null   object 
 7   Grade       1422 non-null   object 
 8   Company     1422 non-null   object 
 9   City        1356 non-null   object 
 10  Skill       880 non-null    object 
 11  Industry    1422 non-null   object 
 12  SourceType  0 non-null      float64
 13  Created     1422 non-null   object 
dtypes: float64(3), int64(3), object(8)
memory usage: 155.7+ KB
None


In [18]:
print("\n=== ПРОПУЩЕННЫЕ ЗНАЧЕНИЯ ===")
print(df.isnull().sum())


=== ПРОПУЩЕННЫЕ ЗНАЧЕНИЯ ===
Value            0
Quarter          0
Year             0
Age            781
Gender         486
Started        784
Profession      37
Grade            0
Company          0
City            66
Skill          542
Industry         0
SourceType    1422
Created          0
dtype: int64


In [ ]:
print("\n=== ПРОВЕРКА БЫВШЕЙ ПРОБЛЕМНОЙ СТРОКИ ===")
if len(df) >= 18:
    print("Строка 18 (индекс 17):")
    print(df.iloc[17])
else:
    print("В датафрейме меньше 18 строк")

# (FMCG; Retail) как мы видим, все ок


=== ПРОВЕРКА БЫВШЕЙ ПРОБЛЕМНОЙ СТРОКИ ===
Строка 18 (индекс 17):
Value                          900000
Quarter                             3
Year                             2025
Age                              25.0
Gender                 PreferNotToSay
Started                        2019.0
Profession        Fullstack Developer
Grade                          Middle
Company                         Local
City                           Almaty
Skill                         Laravel
Industry      Торговля (FMCG; Retail)
SourceType                        NaN
Created           2025-08-21 05:15:46
Name: 17, dtype: object


#  Для практики SQL, давайте загрузим предочищенный датасет в DuckDB

In [2]:
# Загружаем сразу предочищенные данные, так как инструментарий pandas лучше в этом плане

# так как проект больше для практики, загружаем временное хранилище.
# Запуск базы данных либо поднять контейнер постгре ест не малые ресурсы
conn = duckdb.connect(':memory:')

# Регистрируем df как таблицу
conn.register("salaries", df)

# Делаем SQL-запрос
result = conn.execute("""--sql
SELECT *
FROM (
    SELECT
        ROW_NUMBER() OVER () AS rn, s.*
    FROM salaries s) t
WHERE rn = 18;
""").fetchdf()

result

NameError: name 'df' is not defined

In [ ]:
print("=== ДЕТАЛЬНАЯ ИНФОРМАЦИЯ ОБ ОБЪЕКТАХ ===")
detailed_info = conn.execute("""--sql
    SELECT
        table_name,
        table_type,
        table_schema
    FROM information_schema.tables
    WHERE table_name = 'salaries'
""").fetchdf()
print(detailed_info)

In [ ]:
print("\n=== ЕСЛИ НУЖНА РЕАЛЬНАЯ ТАБЛИЦА ===")

# Если хотите реальную таблицу вместо VIEW:
conn.execute("DROP VIEW IF EXISTS salaries")

# Создаем настоящую таблицу
conn.execute("CREATE TABLE salaries AS SELECT * FROM df")

# Проверяем
table_type = conn.execute("""--sql
    SELECT table_type
    FROM information_schema.tables
    WHERE table_name = 'salaries'
""").fetchone()[0]

print(f"Теперь это: {table_type}")

In [ ]:

# 1. В DuckDB для описания данных можно использовать функцию `DESCRIBE`

print(conn.execute("DESCRIBE salaries").fetchdf())

In [ ]:
# 1. Классическое описание в pandas (здесь можно заметить что поле sorceType пустой)
print(df.info())

In [ ]:
# 2. # По колонкам
conn.execute("""--sql
    SELECT
        SUM(CASE WHEN Value IS NULL THEN 1 ELSE 0 END) AS Value_nulls,
        SUM(CASE WHEN Quarter IS NULL THEN 1 ELSE 0 END) AS Quarter_nulls,
        SUM(CASE WHEN Year IS NULL THEN 1 ELSE 0 END) AS Year_nulls,
        SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS Age_nulls,
        SUM(CASE WHEN Gender IS NULL THEN 1 ELSE 0 END) AS Gender_nulls,
        SUM(CASE WHEN Started IS NULL THEN 1 ELSE 0 END) AS Started_nulls,
        SUM(CASE WHEN Profession IS NULL THEN 1 ELSE 0 END) AS Profession_nulls,
        SUM(CASE WHEN Grade IS NULL THEN 1 ELSE 0 END) AS Grade_nulls,
        SUM(CASE WHEN Company IS NULL THEN 1 ELSE 0 END) AS Company_nulls,
        SUM(CASE WHEN City IS NULL THEN 1 ELSE 0 END) AS City_nulls,
        SUM(CASE WHEN Skill IS NULL THEN 1 ELSE 0 END) AS Skill_nulls,
        SUM(CASE WHEN Industry IS NULL THEN 1 ELSE 0 END) AS Industry_nulls,
        SUM(CASE WHEN SourceType IS NULL THEN 1 ELSE 0 END) AS SourceType_nulls,
        SUM(CASE WHEN Created IS NULL THEN 1 ELSE 0 END) AS Created_nulls
    FROM salaries
""").fetchdf()



In [ ]:
# 2. Подсчёт null по колонкам
df.isnull().sum()

In [ ]:
print("=== ВСЕ ТАБЛИЦЫ И VIEWS ===")
objects = conn.execute("SHOW TABLES").fetchdf()
print(objects)

In [ ]:
print("=== ДЕТАЛЬНАЯ ИНФОРМАЦИЯ ОБ ОБЪЕКТАХ ===")
detailed_info = conn.execute("""--sql
    SELECT
        table_name,
        table_type,
        table_schema
    FROM information_schema.tables
    WHERE table_name = 'salaries'
""").fetchdf()
print(detailed_info)

In [ ]:
# 3. Попробуем удалить столбец напрямую
try:
    conn.execute("ALTER TABLE salaries DROP COLUMN SourceType")
    print("✅ Столбец удален через ALTER TABLE")
except Exception as e:
    print(f"❌ ALTER TABLE не сработал: {e}")

# Проверяем результат
print(conn.execute("DESCRIBE salaries").fetchdf())

In [ ]:
# 3. Удаляем колонку sourcetype через Pandas
df = df.drop(columns=["SourceType"])
df.head()

In [ ]:
print("=== ПРОВЕРКА ДУБЛИКАТОВ ЧЕРЕЗ DUCKDB ===")

# Общее количество строк
total_rows = conn.execute("SELECT COUNT(*) FROM salaries").fetchone()[0]
print(f"Общее количество строк: {total_rows}")

# Количество уникальных строк
unique_rows = conn.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT DISTINCT *
        FROM salaries) t
""").fetchone()[0]
print(f"Уникальных строк: {unique_rows}")

# Количество дубликатов
duplicates_count = total_rows - unique_rows
print(f"Дубликатов: {duplicates_count}")

if duplicates_count > 0:
    print(f"Процент дубликатов: {duplicates_count/total_rows*100:.2f}%")

In [ ]:
print("\n=== АЛЬТЕРНАТИВНЫЙ СПОСОБ ЧЕРЕЗ ROW_NUMBER() ===")

# Получаем список всех колонок в таблице salaries
columns = [col[1] for col in conn.execute("PRAGMA table_info(salaries)").fetchall()]

# Создаём строку через запятую для PARTITION BY
columns_str = ", ".join(columns)

print("Будем группировать по колонкам:", columns_str)


# Используем ROW_NUMBER() для поиска дубликатов
duplicate_analysis = conn.execute(f"""--sql
    WITH numbered_rows AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY {columns_str} ORDER BY {columns[0]}) as rn
        FROM salaries
    )
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN rn > 1 THEN 1 ELSE 0 END) as duplicate_rows
    FROM numbered_rows
""").fetchone()

total_rows_rn = duplicate_analysis[0]
duplicates_rn = duplicate_analysis[1]
unique_rows_rn = total_rows_rn - duplicates_rn

print(f"Общее количество строк: {total_rows_rn}")
print(f"Дублированных строк: {duplicates_rn}")
print(f"Уникальных строк: {unique_rows_rn}")

In [ ]:
# Все дубликаты
all_duplicates = df[df.duplicated(keep=False)]

# Сортируем по всем колонкам, чтобы одинаковые шли подряд
all_duplicates_sorted = all_duplicates.sort_values(by=list(df.columns))

all_duplicates_sorted


In [ ]:
print("\n=== ПРОВЕРКА ЧЕРЕЗ PANDAS ===")

total = len(df)
duplicates = df.duplicated().sum()
unique = len(df.drop_duplicates())

print(f"общее количество:    {total}")
print(f"дубликаты:          {duplicates}")
print(f"уникальные:         {unique}")

if duplicates > 0:
    print(f"Процент дубликатов: {duplicates/total*100:.2f}%")

    # Показать первые дубликаты
    print("\nПример дублированных строк:")
    duplicated_df = df[df.duplicated(keep=False)]

duplicated_df.head(5)

In [ ]:
print("=== МЕТОД 1: УДАЛЕНИЕ ДУБЛИКАТОВ ЧЕРЕЗ PANDAS ===")

# Работаем только с pandas DataFrame
print(f"Pandas - до удаления: {len(df)} строк")
print(f"Pandas - дубликатов найдено: {df.duplicated().sum()}")

# Удаляем дубликаты в pandas
df_pandas_clean = df.drop_duplicates()

print(f"Pandas - после удаления: {len(df_pandas_clean)} строк")
print(f"Pandas - удалено: {len(df) - len(df_pandas_clean)} строк")

# Проверяем результат pandas
pandas_duplicates_after = df_pandas_clean.duplicated().sum()
print(f"Pandas - дубликатов осталось: {pandas_duplicates_after}")

# Показываем статистику
print(f"\nРезультат pandas метода:")
print(f"  Уникальных строк: {len(df_pandas_clean)}")

In [ ]:
df = df_pandas_clean

In [ ]:

#  Посмотрим количество строк до удаления
before_count = conn.execute("SELECT COUNT(*) FROM salaries").fetchone()[0]

#  Удалим дубликаты, оставим только уникальные строки
conn.execute("""--sql
CREATE OR REPLACE TABLE salaries AS
SELECT DISTINCT *
FROM salaries
""")

#  Посмотрим количество строк после удаления
after_count = conn.execute("SELECT COUNT(*) FROM salaries").fetchone()[0]

print(f"До удаления: {before_count}")
print(f"После удаления: {after_count}")
print(f"Удалено: {before_count - after_count}")


**После очистки дубликатов и пустых столбцов, нужно подумать что делать с пропущенными значениями**

In [ ]:
df.isnull().sum()

In [ ]:
# запрос на подсчет NULL по всем нужным колонкам
query = """--sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(Value) AS Value_nulls,
    COUNT(*) - COUNT(Quarter) AS Quarter_nulls,
    COUNT(*) - COUNT(Year) AS Year_nulls,
    COUNT(*) - COUNT(Age) AS Age_nulls,
    COUNT(*) - COUNT(Gender) AS Gender_nulls,
    COUNT(*) - COUNT(Started) AS Started_nulls,
    COUNT(*) - COUNT(Profession) AS Profession_nulls,
    COUNT(*) - COUNT(Grade) AS Grade_nulls,
    COUNT(*) - COUNT(Company) AS Company_nulls,
    COUNT(*) - COUNT(City) AS City_nulls,
    COUNT(*) - COUNT(Skill) AS Skill_nulls,
    COUNT(*) - COUNT(Industry) AS Industry_nulls,
    COUNT(*) - COUNT(Created) AS Created_nulls
FROM salaries
"""
result = conn.execute(query).df()
result


**Нам придется удалить строки без названия должности, так как никакой пользы для нашей выборки эти данные не имеют**

In [ ]:
df_null_prof = df[df["Profession"].isna()].head(5)

df_null_prof

In [ ]:
# фильтруем строки, где Profession пустой
df_null_prof = df[df["Profession"].isna()]

# считаем, сколько среди них пустых и непустых Skill
skill_check = df_null_prof["Skill"].isna().value_counts()

print("Проверка по строкам с пустым Profession:")
print(skill_check)

In [ ]:
# как мы видим, где не указана должность так же не указаны и скил или техстек.
# В связи с этим, удаляем ненужные записи

# удаляем строки где Profession == NaN
df_clean = df.dropna(subset=["Profession"])

print(f"Размер до очистки: {df.shape[0]} строк")
print(f"Размер после очистки: {df_clean.shape[0]} строк")

# посмотрим первые 5 строк
df_clean.head()

In [ ]:
df = df_clean

In [ ]:
# создаем очищенную таблицу без NULL в Profession
conn.execute("""--sql
    CREATE OR REPLACE TABLE salaries AS
    SELECT *
    FROM salaries
    WHERE Profession IS NOT NULL
""")

# проверим количество строк
row_count = conn.execute("SELECT COUNT(*) FROM salaries").fetchone()[0]
print(f"После удаления: {row_count} строк")

# посмотрим первые строки
conn.execute("SELECT * FROM salaries LIMIT 5").df()


In [ ]:

# Найти дублирующиеся строки в таблице salaries с помощью DuckDB
duplicates_df = conn.execute("""--sql
    SELECT
        *,
        COUNT(*) as duplicate_count
    FROM salaries
    GROUP BY
        Value, Quarter, Year, Age, Gender, Started, Profession, Grade, Company, City, Skill, Industry, Created
    HAVING COUNT(*) > 1
""").df()

duplicates_df

In [ ]:
# Найти дублирующиеся строки в df по всем колонкам и посчитать их количество
duplicates_df = df[df.duplicated(keep=False)]

# Группируем и считаем количество повторов для каждой дублирующейся строки
duplicates_grouped = (
    duplicates_df
    .groupby(list(df.columns))
    .size()
    .reset_index(name='duplicate_count')
    .query('duplicate_count > 1')
)

duplicates_grouped

In [ ]:
df.head()

In [ ]:
# Получить вторую по величине зарплату (Value) через DuckDB
second_highest_salary = conn.execute("""--sql
    SELECT
        DISTINCT Value
    FROM salaries
    ORDER BY Value DESC
    LIMIT 1
    OFFSET 1
""").fetchone()[0]

print(f"Вторая по величине зарплата: {second_highest_salary}")

In [ ]:
second_highest_salary_pandas = df['Value'].nlargest(2).iloc[-1]
print(f"Вторая по величине зарплата (pandas): {second_highest_salary_pandas}")

In [ ]:
# another way
sec__h_s = conn.execute("""--sql
SELECT Profession, Grade, Company, City, Skill, Age, Gender, Value
FROM (
    SELECT *,
           DENSE_RANK() OVER (ORDER BY Value DESC) AS rnk
    FROM salaries
) t
WHERE rnk = 2
ORDER BY Value DESC;
""").fetchdf()

sec__h_s

In [ ]:
second_salary = df["Value"].nlargest(2).iloc[-1]
top2 = df[df["Value"] == second_salary].sort_values("Value", ascending=False)
top2.head(20)


In [ ]:
df.info()

In [ ]:
# 1. Средний по (Gender + Profession + Grade)
df["Age"] = df["Age"].fillna(
    df.groupby(["Gender", "Profession", "Grade"])["Age"].transform("mean")
)

# 2. Если NaN остались → средний по (Profession + Grade)
df["Age"] = df["Age"].fillna(
    df.groupby(["Profession", "Grade"])["Age"].transform("mean")
)

# 3. Если всё ещё NaN → средний по Gender
df["Age"] = df["Age"].fillna(
    df.groupby("Gender")["Age"].transform("mean")
)

# 4. И в крайнем случае — общий средний
df["Age"] = df["Age"].fillna(df["Age"].mean())

print("Остаток NaN:", df["Age"].isna().sum())


In [ ]:
df.info()

In [ ]:
df["City"] = df["City"].astype("string")

# Заполняем NaN самым частым городом в группе (Company + Profession + Grade)
df["City"] = df.groupby(["Company", "Profession", "Grade"])["City"] \
               .transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan))

# Если остались NaN → заполняем по компании
df["City"] = df.groupby("Company")["City"] \
               .transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan))

# Если ещё остались → глобальной модой
df["City"] = df["City"].fillna(df["City"].mode().iloc[0])


In [ ]:
df.info()

In [ ]:
def fill_gender_proportionally(group):
    """Заполняет пропуски в Gender по пропорциям внутри группы"""
    if group["Gender"].isna().sum() == 0:
        return group

    counts = group["Gender"].value_counts(normalize=True)
    if counts.empty:
        return group

    missing = group["Gender"].isna()
    group.loc[missing, "Gender"] = np.random.choice(
        counts.index,
        size=missing.sum(),
        p=counts.values
    )
    return group

# 1. Сначала пробуем заполнить по (Profession + Grade)
df = df.groupby(["Profession", "Grade"], group_keys=False).apply(fill_gender_proportionally)

# 2. Если остались NaN → по Profession
df = df.groupby("Profession", group_keys=False).apply(fill_gender_proportionally)

# 3. Если ещё остались → по общему распределению
if df["Gender"].isna().sum() > 0:
    counts = df["Gender"].value_counts(normalize=True)
    missing = df["Gender"].isna()
    df.loc[missing, "Gender"] = np.random.choice(
        counts.index,
        size=missing.sum(),
        p=counts.values
    )

print("Остаток NaN:", df["Gender"].isna().sum())
print("Распределение полов:\n", df["Gender"].value_counts(normalize=True))


In [ ]:
df.info()

In [3]:
df = df.rename(columns={
    "Company": "Localization",
    "Value": "Salary"})

NameError: name 'df' is not defined

In [ ]:
# Перерегистрируем очищенный df как новую таблицу
conn.register("salaries_clean", df)

# Перезаписываем старую таблицу
conn.execute("DROP TABLE IF EXISTS salaries")
conn.execute("CREATE TABLE salaries AS SELECT * FROM salaries_clean")

In [ ]:
query = "SELECT * FROM salaries LIMIT 5"
result = conn.execute(query).fetchdf()
result

In [ ]:
df["Age"] = df["Age"].round().astype(int)

In [ ]:
# Save cleaned DataFrame to CSV
df.to_csv("salaries_clean.csv", index=False)